<a id="top"></a>
# Референсный ноутбук: BRFSS 2024 · NHANES 2021–2023 · AI4FoodDB

Шаблон анализа для темы «Привычки и здоровье». Не отчёт, а каркас: порядок шагов,
проверки и правила, которые команда копирует в свой ноутбук и наполняет выводами.

**Как читать.** Пошаговая инструкция по этому ноутбуку: [гид по ноутбуку](T1-MEMO_NOTEBOOK_GUIDE.html) — 12 шагов с вопросами, проверками и примерами кода.

**Что внутри**

1. цель исследования и проблема, которую он решает;
2. общий обзор исходных данных с визуализацией;
3. паспорт данных и проверка целостности;
4. загрузка трёх наборов и разбор ловушек кодов;
5. гармонизация общих понятий (ИМТ, сон, активность, рацион, дистресс);
6. QC: пропуски, диапазоны, выбросы, дубликаты;
7. описательные таблицы, взвешенные и невзвешенные;
8. модель: важность блоков привычек с bootstrap-CI;
9. метаанализ: эффект по наборам, fixed/random, I², forest plot;
10. кластерный анализ: когорты и области на скаттер-диаграмме;
11. стандарт визуализации;
12. документирование и воспроизводимость;
13. выводы: как результат решает исходную проблему.

**Как запускать**

```
cd analysis/t1
jupyter lab notebooks/REFERENCE_ANALYSIS.ipynb
```

Ноутбук ищет корень `analysis/t1` подъёмом вверх от текущей папки, поэтому его можно
запускать из `notebooks/` или из корня. Данные только читаются: `processed/` не меняется,
все артефакты пишутся в `out/`.

**Лицензии.** BRFSS и NHANES — public domain (CDC). AI4FoodDB — CC BY-NC-ND 4.0:
только некоммерческое использование, производные распространять нельзя. В ноутбуке
AI4FoodDB используется как проверка схемы (один участник), а не как популяция.

**Правило.** Если шаг нельзя воспроизвести одной командой — он не готов.

## Содержание

- [1. Цель и проблема](#sec-1)
- [2. Обзор исходных данных](#sec-2)
- [3. Паспорт данных и целостность](#sec-3)
- [4. BRFSS 2024: взвешенные оценки и ловушки кодов](#sec-4)
- [5. NHANES 2021–2023: сборка когорты](#sec-5)
- [6. AI4FoodDB: проверка схемы, не данные](#sec-6)
- [7. Гармонизация: общие понятия и разные вопросы](#sec-7)
- [7.1. Главные компоненты и корреляции по наборам](#sec-7-1)
- [8. QC: пропуски, диапазоны, выбросы](#sec-8)
- [9. Модель: важность блоков привычек](#sec-9)
- [10. Метаанализ: эффект по наборам](#sec-10)
- [11. Кластерный анализ: когорты по привычкам](#sec-11)
- [12. Стандарт визуализации](#sec-12)
- [13. Документирование и воспроизводимость](#sec-13)
- [14. Чек-лист перед сдачей](#sec-14)
- [15. Выводы: как исследование решает проблему](#sec-15)

[↑ в начало](#top)


<a id="sec-1"></a>
## 1. Цель и проблема

**Проблема.** Решения о питании и профилактике опираются на привычки, но данные о них
разрознены: телефонный опрос (BRFSS), обследование с измерениями (NHANES) и носимые
устройства (AI4FoodDB) отвечают на разные вопросы и не совпадают по шкалам. Из-за этого
нельзя честно сравнить вклад сна, активности, рациона и дистресса в ИМТ и понять, какие
привычки связаны с риском, а какие — только с возрастом.

**Цель.** Собрать три источника в один воспроизводимый конвейер и ответить на вопрос:
какие блоки привычек объясняют ИМТ сверх демографии, устойчив ли эффект между наборами
и какие когорты людей различаются по профилю привычек.

**Задачи.**

1. Проверить источники и собрать паспорт: строки, колонки, лицензии, кодовые ловушки.
2. Сгармонизировать понятия: ИМТ, сон, активность, рацион, дистресс.
3. Провести QC и описательные таблицы, взвешенные и невзвешенные.
4. Оценить важность блоков привычек с bootstrap-CI и проверить устойчивость на seed'ах.
5. Сравнить эффект между наборами и выделить когорты кластерным анализом.
6. Сформулировать выводы и ограничения; показать, как результат помогает продукту.

**Что считается результатом.** Ноутбук, который прогоняется сверху вниз без ручных
шагов, плюс метрики, факторы с CI, графики и выводы-гипотезы.

**Гипотезы.** H1: активность связана с ИМТ сильнее, чем сон и дистресс. H2: направление
связи совпадает в BRFSS и NHANES. H3: когорты по привычкам различаются возрастом и ИМТ,
а не только рационом.

In [ ]:
from __future__ import annotations

import json
import math
import platform
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats
from scipy.spatial import ConvexHull
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:  # запуск как скрипта, без Jupyter
    display = print

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# Корень t1: ищем scripts/ в текущей папке, выше или в подпапке t1.
START = Path.cwd().resolve()
ROOT = START
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "scripts").is_dir() and (START / "t1" / "scripts").is_dir():
    ROOT = START / "t1"
assert (ROOT / "scripts").is_dir(), "откройте ноутбук из папки t1"

PROCESSED = ROOT / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
RAW = ROOT / "raw"
OUT = ROOT / "out"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> Path:
    """Сохранить график в out/figures и вернуть путь."""
    path = FIG / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    return path

CACHE = PROCESSED / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
COHORT_PATH = CACHE / "cohort_nhanes.parquet"
COHORT_META = CACHE / "cohort_nhanes.json"

brfss_path = PROCESSED / "brfss2024.parquet"
nhanes_dir = PROCESSED / "nhanes"
nhanes_files = sorted(nhanes_dir.glob("*_L.parquet"))

NHANES_MODULES = ("DEMO", "BMX", "BPXO", "SLQ", "PAQ", "DR1TOT", "DPQ")
CONVERTERS = {
    "BRFSS 2024": "scripts/convert_brfss.py",
    "NHANES": "scripts/convert_nhanes.py",
    "AI4FoodDB": "scripts/convert_ai4food.py",
}


def data_status() -> pd.DataFrame:
    """Что уже собрано в processed/: готовые паркеты берём, недостающие — собираем."""
    rows = []

    def add(block: str, path: Path, hint: str) -> None:
        rows.append({"блок": block, "путь": str(path.relative_to(ROOT)),
                     "есть": path.exists(),
                     "что делать": "" if path.exists() else hint})

    add("BRFSS 2024", brfss_path, "python scripts/convert_brfss.py")
    for m in NHANES_MODULES:
        add(f"NHANES {m}", nhanes_dir / f"{m}_L.parquet", "python scripts/convert_nhanes.py")
    add("AI4FoodDB", PROCESSED / "ai4fooddb", "python scripts/convert_ai4food.py")
    add("Когорта NHANES (кэш)", COHORT_PATH, "соберётся в разделе 5")
    return pd.DataFrame(rows)


def ensure_processed(auto: bool = True) -> pd.DataFrame:
    """Готовые паркеты используем как есть; конвертеры — только для недостающих.

    Если parquet уже собран (после первого запуска), повторный запуск не тратит
    время на конвертацию: проверка занимает миллисекунды.
    """
    status = data_status()
    missing = status[(~status["есть"]) & (status["блок"] != "Когорта NHANES (кэш)")]
    if missing.empty:
        print("processed/: всё на месте — берём готовые паркеты")
        return status
    print("не хватает:", ", ".join(missing["блок"]))
    if not auto:
        raise FileNotFoundError("нет паркетов: запустите scripts/convert_*.py")
    need = set()
    for block in missing["блок"]:
        if block.startswith("BRFSS"):
            need.add("BRFSS 2024")
        elif block.startswith("NHANES"):
            need.add("NHANES")
        else:
            need.add("AI4FoodDB")
    for key in need:
        script = ROOT / CONVERTERS[key]
        if not script.exists():
            raise FileNotFoundError(f"нет конвертера {script}")
        print("запускаю", script.name, "…")
        subprocess.run([sys.executable, str(script)], cwd=ROOT, check=True)
    print("готово: паркеты собраны")
    return data_status()


if (ROOT / "raw").is_dir() and any((ROOT / "raw").iterdir()):
    status = ensure_processed()
else:
    status = data_status()
    print("raw/ не найден: данные не загружены. Как скачать — в 01_opendata_research/README.md.")
display(status.style.apply(
    lambda r: ["background:#EAF3E3" if r["есть"] else "background:#FDECEC"] * len(r), axis=1))

SEEDS = [42, 7, 2024]
PALETTE = {
    "violet": "#4B3F82", "magenta": "#E62E7C", "olive": "#6F9C4A",
    "coral": "#F57D64", "lav": "#9B8ED6", "grey": "#6E6690",
    "sage": "#B2BE92", "dark": "#2E2657",
}
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 160, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#EDE7FB", "grid.linewidth": 0.8,
    "axes.titleweight": "bold", "axes.titlesize": 11,
    "figure.facecolor": "white",
})

print("корень:", ROOT)
print("python:", platform.python_version(), "| pandas", pd.__version__,
      "| numpy", np.__version__, "| sklearn", __import__("sklearn").__version__)

<a id="sec-2"></a>
## 2. Обзор исходных данных

Три источника разного типа и разного доверия:

| Тип | Источник | Что даёт | Ограничение |
|---|---|---|---|
| телефонный опрос | BRFSS 2024 | масштаб, взвешенные оценки | самоотчёт, нет измерений |
| обследование | NHANES 2021–2023 | измерения, 24h recall | меньше выборка, цикл L |
| носимые устройства | AI4FoodDB | пульс, сон, активность | один участник — схема, не данные |

Ниже — карта данных: объёмы таблиц, распределения целевых показателей и доля непустых
значений. По ней видно, что сравнивать, а что нельзя.

In [ ]:
def table_size(path: Path) -> dict:
    """Строки, колонки и размер файла — для карты источников."""
    schema = pq.read_schema(path)
    return {"rows": pq.read_metadata(path).num_rows, "cols": len(schema.names),
            "mb": round(path.stat().st_size / 1e6, 2)}

sources = {"BRFSS 2024": brfss_path}
for p in nhanes_files:
    sources[f"NHANES {p.stem.replace('_L', '')}"] = p
overview = pd.DataFrame([{"источник": k, **table_size(v)} for k, v in sources.items()])
overview = overview.sort_values("rows", ascending=False)
print("таблиц в обзоре:", len(overview), "| строк всего:",
      f"{overview['rows'].sum():,}".replace(",", " "))
overview.head(10)

In [ ]:
# Четыре панели: объём таблиц, ИМТ в двух наборах, возраст NHANES, доля непустых.
fig, axes = plt.subplots(2, 2, figsize=(11.5, 6.6))

ax = axes[0, 0]
top = overview.head(10).iloc[::-1]
ax.barh(top["источник"], top["rows"], color=PALETTE["violet"])
ax.set_xscale("log")
ax.set_xlabel("строк (логарифмическая шкала)")
ax.set_title("Объём таблиц")
ax.grid(axis="y", visible=False)

ax = axes[0, 1]
bmi_b = pq.read_table(brfss_path, columns=["_BMI5"]).to_pandas()["_BMI5"]
bmi_b = (bmi_b.where(bmi_b.between(1000, 7000)) / 100).dropna()
bmi_n = pq.read_table(nhanes_dir / "BMX_L.parquet", columns=["BMXBMI"]).to_pandas()["BMXBMI"].dropna()
bins = np.arange(14, 51, 1)
ax.hist(bmi_b.clip(14, 50), bins=bins, density=True, alpha=.5,
        color=PALETTE["violet"], label=f"BRFSS (n={len(bmi_b):,})".replace(",", " "))
ax.hist(bmi_n.clip(14, 50), bins=bins, density=True, alpha=.5,
        color=PALETTE["magenta"], label=f"NHANES (n={len(bmi_n):,})".replace(",", " "))
ax.set_xlabel("ИМТ, кг/м²")
ax.set_ylabel("плотность")
ax.set_title("ИМТ: опрос против обследования")
ax.legend(fontsize=8)

ax = axes[1, 0]
age = pq.read_table(nhanes_dir / "DEMO_L.parquet", columns=["RIDAGEYR"]).to_pandas()["RIDAGEYR"]
ax.hist(age, bins=range(0, 85, 5), color=PALETTE["olive"], alpha=.85)
ax.set_xlabel("возраст, лет")
ax.set_ylabel("человек")
ax.set_title("NHANES: возраст участников")

ax = axes[1, 1]
pairs = [
    ("BRFSS: ИМТ", brfss_path, "_BMI5"),
    ("BRFSS: активность", brfss_path, "_TOTINDA"),
    ("NHANES: ИМТ", nhanes_dir / "BMX_L.parquet", "BMXBMI"),
    ("NHANES: сон", nhanes_dir / "SLQ_L.parquet", "SLD012"),
    ("NHANES: PHQ-9", nhanes_dir / "DPQ_L.parquet", "DPQ010"),
    ("NHANES: ккал", nhanes_dir / "DR1TOT_L.parquet", "DR1TKCAL"),
]
share = []
for label, path, col in pairs:
    col_data = pq.read_table(path, columns=[col]).column(0).to_pandas()
    share.append({"переменная": label, "доля": float(col_data.notna().mean())})
share = pd.DataFrame(share)
ax.barh(share["переменная"], share["доля"], color=PALETTE["lav"])
ax.set_xlim(0, 1)
ax.set_xlabel("доля непустых")
ax.set_title("Заполненность ключевых переменных")
ax.grid(axis="y", visible=False)
for y, v in enumerate(share["доля"]):
    ax.text(min(v + 0.02, 0.98), y, f"{v:.0%}", va="center", fontsize=8)

fig.tight_layout()
fig.savefig(FIG / "data_overview.png", bbox_inches="tight")
plt.show()
print("сохранено:", FIG / "data_overview.png")

<a id="sec-3"></a>
## 3. Паспорт данных и целостность

Перед любым анализом проверяем, что файлы на месте, размеры совпадают с паспортом,
а sha256 сырья не изменился. Это дешевле, чем искать причину странных чисел в конце.

Правила:

- числа строк × колонок сверяем с `PASSPORT.md`, а не «на глаз»;
- конвертация пишет манифест (`out/*_convert.json`) — читаем его, а не угадываем;
- если файла нет, ноутбук должен падать с понятным сообщением, а не молча считать пустоту.

In [ ]:
manifest = pd.read_csv(OUT / "manifest_raw.csv")
print("строк в манифесте:", len(manifest))
manifest.head(8)

conv = {}
for name in ("brfss2024", "nhanes", "ai4fooddb"):
    p = OUT / f"{name}_convert.json"
    if p.exists():
        conv[name] = json.loads(p.read_text(encoding="utf-8"))
print("манифесты конвертации:", list(conv))

brfss_path = ROOT / "processed" / "brfss2024.parquet"
assert brfss_path.exists(), f"нет {brfss_path} — сначала scripts/convert_brfss.py"
nhanes_dir = ROOT / "processed" / "nhanes"
nhanes_files = sorted(nhanes_dir.glob("*_L.parquet"))
print("BRFSS:", round(brfss_path.stat().st_size / 1e6, 1), "МБ")
print("NHANES модулей:", len(nhanes_files), [p.name.replace("_L.parquet", "") for p in nhanes_files])

In [ ]:
# Паспорт: строки против колонок, размер круга — мегабайты parquet.
fig, ax = plt.subplots(figsize=(7.4, 4.2))
sizes = (overview["mb"].clip(lower=0.01) ** 0.5) * 34
ax.scatter(overview["rows"], overview["cols"], s=sizes, alpha=.55,
           color=PALETTE["violet"], edgecolor="white", linewidth=1)
for _, r in overview.iterrows():
    ax.annotate(r["источник"].replace("NHANES ", ""), (r["rows"], r["cols"]),
                textcoords="offset points", xytext=(7, 4), fontsize=8,
                color=PALETTE["grey"])
ax.set_xscale("log")
ax.set_xlabel("строк (логарифмическая шкала)")
ax.set_ylabel("колонок")
ax.set_title("Паспорт: объём и ширина таблиц")
fig.tight_layout()
save_fig(fig, "passport_shape")
plt.show()

<a id="sec-4"></a>
## 4. BRFSS 2024: взвешенные оценки и ловушки кодов

BRFSS — телефонный опрос со сложным дизайном: без веса `_LLCPWT` средние смещены.
Коды «не знаю / отказ» (7, 9, 77, 9999) нельзя оставлять числами — они ломают средние.

Ловушки, проверенные по паспорту:

- `_BMI5` — ИМТ × 100, код 9999 = отказ;
- вес для анализа — `_LLCPWT` (есть у всех), а не `_CLLCPWT` (12 %);
- `_PHYS14D` — дни плохого физического здоровья, это **не** физическая активность;
- в файле 2024 **нет** `SLEPTIM1`, `STRESS`, `PULSE`, `_FRTLT1`, `_VEGLT1`: сон,
  стресс, пульс и частота фруктов-овощей в этом году не спрашивались. Это проверено
  по схеме parquet, а не по памяти. Поэтому общий эффект с NHANES строим по активности.

In [ ]:
# Сначала проверяем, что переменные реально есть в файле: кодбук и файл расходятся.
BRFSS_COLS = ["_BMI5", "_LLCPWT", "_AGEG5YR", "_SEX", "_EDUCAG", "_INCOMG1",
              "GENHLTH", "_TOTINDA", "EXERANY2", "SMOKE100", "_DRNKWK3", "MENTHLTH"]
present = set(pq.read_schema(brfss_path).names)
absent = [c for c in BRFSS_COLS if c not in present]
assert not absent, f"в BRFSS 2024 нет колонок: {absent}"

brfss = pq.read_table(brfss_path, columns=BRFSS_COLS).to_pandas()
brfss["bmi"] = brfss["_BMI5"].where(brfss["_BMI5"].between(1000, 7000)) / 100
brfss["w"] = brfss["_LLCPWT"]
brfss["active"] = brfss["_TOTINDA"].map({1: 1.0, 2: 0.0})

def wmean(x: pd.Series, w: pd.Series) -> float:
    """Взвешенное среднее; строки без веса и без значения не учитываются."""
    m = x.notna() & w.notna() & (w > 0)
    return float((x[m] * w[m]).sum() / w[m].sum()) if m.any() else float("nan")

def wshare(mask: pd.Series, w: pd.Series) -> float:
    """Взвешенная доля; mask — булев ряд."""
    m = mask.notna() & w.notna() & (w > 0)
    return float(w[m][mask[m]].sum() / w[m].sum()) if m.any() else float("nan")

print("строк:", len(brfss), "| колонок прочитано:", brfss.shape[1])
print("вес _LLCPWT заполнен:", f"{brfss['w'].notna().mean():.1%}")
print("ИМТ: невзвешенный", round(brfss['bmi'].mean(), 2),
      "| взвешенный", round(wmean(brfss['bmi'], brfss['w']), 2))
print("нет активности:", f"{wshare(brfss['active'] == 0, brfss['w']):.1%}")
print("ожирение (ИМТ 30+):", f"{wshare(brfss['bmi'] >= 30, brfss['w']):.1%}")
print("дней плохого психического здоровья (среднее):",
      round(wmean(brfss['MENTHLTH'].where(brfss['MENTHLTH'].between(0, 30)), brfss['w']), 2))

In [ ]:
# BRFSS: насколько вес _LLCPWT меняет вывод. ИМТ и доли — на разных осях:
# смешивать кг/м² и проценты на одной шкале нельзя.
bmi_raw = float(brfss["bmi"].mean())
bmi_w = wmean(brfss["bmi"], brfss["w"])
ob_raw = float(brfss["bmi"].ge(30).mean())
ob_w = wshare(brfss["bmi"] >= 30, brfss["w"])
inact_raw = float((brfss["active"] == 0).mean())
inact_w = wshare(brfss["active"] == 0, brfss["w"])

fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.2))
ax = axes[0]
ax.bar(["невзвешенно", "взвешенно"], [bmi_raw, bmi_w],
       color=[PALETTE["lav"], PALETTE["violet"]], width=.55)
for i, v in enumerate([bmi_raw, bmi_w]):
    ax.text(i, v + 0.15, f"{v:.2f}", ha="center", fontsize=9)
ax.set_ylim(0, max(bmi_raw, bmi_w) * 1.25)
ax.set_ylabel("ИМТ, кг/м²")
ax.set_title("ИМТ: вес почти не меняет")

ax = axes[1]
x = np.arange(2)
w = 0.36
ax.bar(x - w / 2, [ob_raw, inact_raw], w, color=PALETTE["lav"], label="невзвешенно")
ax.bar(x + w / 2, [ob_w, inact_w], w, color=PALETTE["violet"], label="взвешенно")
for i, (a, b) in enumerate([(ob_raw, ob_w), (inact_raw, inact_w)]):
    ax.text(i - w / 2, a + .01, f"{a:.2f}", ha="center", fontsize=8)
    ax.text(i + w / 2, b + .01, f"{b:.2f}", ha="center", fontsize=8)
ax.set_xticks(x, ["ожирение", "нет активности"])
ax.set_ylim(0, max(ob_raw, ob_w, inact_raw, inact_w) * 1.35)
ax.set_ylabel("доля")
ax.set_title("Доли: вес меняет сильнее")
ax.legend(fontsize=8, loc="upper right")
fig.tight_layout()
save_fig(fig, "brfss_weighted")
plt.show()

<a id="sec-5"></a>
## 5. NHANES 2021–2023: сборка когорты

NHANES — обследование с измерениями, а не самоотчётом, поэтому это основной источник
для модели. Собираем взрослых 20+, у кого есть измеренный ИМТ и достоверный 24h recall.

Ловушки цикла L:

- давление в `BPXO_*` (осциллометрия), имён `BPXSY1` нет;
- в `SLQ` только часы сна, вопросов про нарушения сна больше нет;
- прямого вопроса про стресс нет — используем PHQ-9 как прокси дистресса;
- `DPQ` коды 7/9 — отказ, их нельзя складывать в сумму.

In [ ]:
def nhanes(mod: str, cols: list[str] | None = None) -> pd.DataFrame:
    """Один модуль NHANES из processed/nhanes."""
    return pq.read_table(nhanes_dir / f"{mod}_L.parquet", columns=cols).to_pandas()

def schema(mod: str) -> list[str]:
    return pq.read_schema(nhanes_dir / f"{mod}_L.parquet").names

# Минуты активности в неделю: частота × дней в неделе × минуты за раз.
UNIT_PER_WEEK = {"D": 7.0, "W": 1.0, "M": 1 / 4.345, "Y": 1 / 52.14}
def weekly_minutes(q, unit, minutes) -> pd.Series:
    """Минуты в неделю из PAQ: частота, единица и длительность; коды 7777/9999 -> NaN."""
    q = q.where(q.between(0, 1000))
    minutes = minutes.where(minutes.between(0, 1440))
    return q * unit.astype("string").map(UNIT_PER_WEEK) * minutes


def cohort_fingerprint() -> str:
    """Отпечаток источников: размер и mtime паркетов, без чтения данных."""
    parts = []
    for p in [brfss_path] + [nhanes_dir / f"{m}_L.parquet" for m in NHANES_MODULES]:
        st = p.stat()
        parts.append(f"{p.name}:{st.st_size}:{int(st.st_mtime)}")
    return "|".join(parts)


def cohort_cache_ok() -> bool:
    """Кэш когорты годен, если паркеты не менялись с момента сборки."""
    if not (COHORT_PATH.exists() and COHORT_META.exists()):
        return False
    try:
        meta = json.loads(COHORT_META.read_text(encoding="utf-8"))
    except ValueError:
        return False
    return meta.get("fingerprint") == cohort_fingerprint()


def build_cohort() -> pd.DataFrame:
    """Собрать когорту из модулей NHANES: взрослые 20+, ИМТ, достоверный recall."""
    demo = nhanes("DEMO", ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "DMDEDUC2",
                           "INDFMPIR", "WTMEC2YR"])
    paq = nhanes("PAQ", ["SEQN", "PAD790Q", "PAD790U", "PAD800",
                         "PAD810Q", "PAD810U", "PAD820"])
    bmx = nhanes("BMX", ["SEQN", "BMXBMI", "BMXWAIST"])
    bp = nhanes("BPXO", ["SEQN", "BPXOSY1", "BPXODI1", "BPXOPLS1"])
    slq = nhanes("SLQ", ["SEQN", "SLD012"])
    dr = nhanes("DR1TOT", ["SEQN", "DR1DRSTZ", "DR1TKCAL", "DR1TPROT", "DR1TCARB",
                           "DR1TTFAT", "DR1TFIBE", "DR1TSODI"])
    dpq_cols = [c for c in schema("DPQ") if c.startswith("DPQ0")]

    nh = (demo.merge(bmx, on="SEQN", how="inner")
              .merge(bp, on="SEQN", how="left")
              .merge(slq, on="SEQN", how="left")
              .merge(dr, on="SEQN", how="left")
              .merge(paq, on="SEQN", how="left")
              .merge(nhanes("DPQ", ["SEQN"] + dpq_cols), on="SEQN", how="left"))

    vig = weekly_minutes(nh["PAD790Q"], nh["PAD790U"], nh["PAD800"])
    mod = weekly_minutes(nh["PAD810Q"], nh["PAD810U"], nh["PAD820"])
    nh["pa_min"] = pd.concat([vig, mod], axis=1).sum(axis=1, min_count=1)
    nh["active"] = (nh["pa_min"] >= 150).astype(float)
    nh.loc[nh["pa_min"].isna(), "active"] = np.nan

    # Коды отказов PHQ-9 (7/9) -> NaN, сумма только при полном наборе ответов.
    dpq_clean = nh[dpq_cols].apply(lambda s: s.where(s.between(0, 3)))
    nh["phq9"] = dpq_clean.sum(axis=1, min_count=len(dpq_cols))
    nh["sleep_h"] = nh["SLD012"].where(nh["SLD012"].between(0, 24))
    nh["bmi"] = nh["BMXBMI"]

    out = nh[(nh["RIDAGEYR"] >= 20) & nh["bmi"].notna()
             & (nh["DR1DRSTZ"] == 1)].copy()
    print("взрослых 20+ с ИМТ:", int(((nh['RIDAGEYR'] >= 20) & nh['bmi'].notna()).sum()))
    print("когорта с достоверным recall:", len(out))
    return out


if cohort_cache_ok():
    cohort = pd.read_parquet(COHORT_PATH)
    print("когорта из кэша:", len(cohort), "· источники не менялись")
else:
    cohort = build_cohort()
    cohort.to_parquet(COHORT_PATH, index=False)
    COHORT_META.write_text(json.dumps({
        "fingerprint": cohort_fingerprint(),
        "rows": int(len(cohort)),
        "created_at": datetime.now().isoformat(timespec="seconds"),
    }, ensure_ascii=False, indent=1), encoding="utf-8")
    print("когорта сохранена в кэш:", COHORT_PATH.relative_to(ROOT))

print(cohort[["RIDAGEYR", "bmi", "sleep_h", "phq9", "DR1TKCAL"]].describe().round(2))

In [ ]:
# Когорта: воронка отбора и распределение активности.
demo_all = nhanes("DEMO", ["SEQN", "RIDAGEYR"])
bmx_all = nhanes("BMX", ["SEQN", "BMXBMI"])
dr_all = nhanes("DR1TOT", ["SEQN", "DR1DRSTZ"])
adults = demo_all[demo_all["RIDAGEYR"] >= 20]
steps = pd.DataFrame({
    "шаг": ["все участники", "взрослые 20+", "есть ИМТ", "достоверный recall", "когорта"],
    "n": [len(demo_all), len(adults),
          int(adults["SEQN"].isin(bmx_all.loc[bmx_all["BMXBMI"].notna(), "SEQN"]).sum()),
          int(adults["SEQN"].isin(dr_all.loc[dr_all["DR1DRSTZ"] == 1, "SEQN"]).sum()),
          len(cohort)],
})
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
ax.barh(steps["шаг"][::-1], steps["n"][::-1], color=PALETTE["violet"])
ax.set_xlabel("человек")
ax.set_title("NHANES: воронка отбора когорты")
ax.grid(axis="y", visible=False)
for y, v in enumerate(steps["n"][::-1]):
    ax.text(v + 60, y, f"{v:,}".replace(",", " "), va="center", fontsize=8)
ax = axes[1]
ax.hist(cohort["pa_min"].clip(0, 900), bins=30, color=PALETTE["olive"], alpha=.85)
ax.set_xlabel("минут активности в неделю (обрезка 900)")
ax.set_ylabel("человек")
ax.set_title("Активность в когорте")
fig.tight_layout()
save_fig(fig, "nhanes_cohort")
plt.show()

<a id="sec-6"></a>
## 6. AI4FoodDB: проверка схемы, не данные

В наборе один участник (`A4F_10021`), 29 таблиц: антропометрия, сон, активность,
питание, биомаркеры, ЭКГ, пульс. Это пример структуры, поэтому:

- использовать для проверки парсеров и схемы гармонизации;
- **не** считать по нему распределения и не включать в метаанализ;
- помнить лицензию CC BY-NC-ND 4.0: производные (в т.ч. parquet) не распространять.

In [ ]:
a4f_dir = ROOT / "processed" / "ai4fooddb"
a4f_files = sorted(a4f_dir.glob("*.parquet"))
print("таблиц:", len(a4f_files))

rows = []
participants = set()
for p in a4f_files:
    schema_a4f = pq.read_schema(p).names
    id_col = next((c for c in ("id", "participant_id", "ParticipantID", "user_id")
                   if c in schema_a4f), None)
    n = pq.read_metadata(p).num_rows
    if id_col:
        participants |= set(pq.read_table(p, columns=[id_col]).column(0).to_pylist())
    rows.append({"table": p.stem, "rows": n, "cols": len(schema_a4f), "id_col": id_col})

a4f_index = pd.DataFrame(rows).sort_values("rows", ascending=False)
print("участников в наборе:", participants)
print("лицензия: CC BY-NC-ND 4.0 — только некоммерческое использование, без производных")
a4f_index.head(10)

In [ ]:
# AI4FoodDB: какие таблицы самые тяжёлые (лог-шкала).
top = a4f_index.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7.6, 4.0))
ax.barh(top["table"], top["rows"], color=PALETTE["coral"])
ax.set_xscale("log")
ax.set_xlabel("строк (логарифмическая шкала)")
ax.set_title("AI4FoodDB: объём таблиц (пример, 1 участник)")
ax.grid(axis="y", visible=False)
fig.tight_layout()
save_fig(fig, "ai4food_tables")
plt.show()

In [ ]:
# Расшифровка таблиц AI4FoodDB: что внутри и в каких единицах.
A4F_DOCS = {
    "DS1": ("антропометрия", "вес, ИМТ, % жира и мышц, талия, бёдра", "кг, кг/м², %, см"),
    "DS2": ("образ жизни и здоровье", "приёмы пищи, вода, алкоголь, курение, стресс и тревога", "мл, порции, баллы"),
    "DS3": ("питание (фото)", "фото блюд, метки и время приёма; макронутриентов нет", "метки, время"),
    "DS4": ("биомаркеры", "глюкоза и лабораторные показатели", "мг/дл и др."),
    "DS5": ("микробиом", "состав кишечной микробиоты", "таксоны, доля"),
    "DS6": ("витальные показатели", "пульс, ЭКГ, давление", "уд/мин, мс, мм рт. ст."),
    "DS7": ("физическая активность", "IPAQ, активные минуты, оценка VO2", "мин/нед, MET, мл/кг/мин"),
    "DS8": ("сон", "опросник Oviedo, часы сна, эффективность", "ч, баллы, %"),
    "DS9": ("эмоциональное состояние", "DASS-21: стресс, тревога, депрессия", "баллы"),
    "DS10": ("дополнительно", "SUS — удовлетворённость приложением", "баллы 0–100"),
}
rows = []
for _, r in a4f_index.iterrows():
    key = r["table"][:3]
    what, inside, unit = A4F_DOCS.get(key, ("—", "—", "—"))
    rows.append({"таблица": r["table"].split("__")[0], "строк": int(r["rows"]),
                 "колонок": int(r["cols"]), "что внутри": what,
                 "показатели": inside, "единицы": unit})
decoded = pd.DataFrame(rows)
print("таблиц:", len(decoded), "· участник один, это пример схемы")
decoded

<a id="sec-7"></a>
## 7. Гармонизация: общие понятия и разные вопросы

| Понятие | BRFSS 2024 | NHANES 2021–2023 | AI4FoodDB |
|---|---|---|---|
| ИМТ | `_BMI5 / 100` | `BMXBMI` (измерен) | `BMI` в DS1 |
| Сон, ч | нет в 2024 | `SLD012` (самоотчёт) | `DS8` (носимое устройство) |
| Активность | `_TOTINDA` (да/нет) | `PAQ` (минуты в неделю) | `DS7` (шаги, минуты) |
| Рацион | нет частотных вопросов | `DR1TOT` (24h recall, граммы) | `DS3` (фото еды) |
| Дистресс | `MENTHLTH` (дни) | PHQ-9 (`DPQ*`) | `DS9` (настроение) |
| Возраст | `_AGEG5YR` (группы) | `RIDAGEYR` (годы) | `age` |

Правило: сравниваем **направление и размер эффекта**, а не абсолютные средние.
Разные инструменты (опросник против носимого устройства) дают разные шкалы.

In [ ]:
def effect_active_bmi(df: pd.DataFrame, active_col: str, bmi_col: str) -> dict:
    """Point-biserial r между активностью (0/1) и ИМТ + размер выборки."""
    d = df[[active_col, bmi_col]].dropna()
    r, p = stats.pointbiserialr(d[active_col], d[bmi_col])
    return {"n": int(len(d)), "r": float(r), "p": float(p)}

effects = {
    "BRFSS 2024": effect_active_bmi(brfss, "active", "bmi"),
    "NHANES 2021–2023": effect_active_bmi(cohort, "active", "bmi"),
}
for name, e in effects.items():
    print(f"{name}: n={e['n']}, r={e['r']:.3f}, p={e['p']:.2g}")

In [ ]:
# Гармонизация: один эффект в двух наборах, с 95 % CI.
fig, ax = plt.subplots(figsize=(6.8, 2.9))
names = list(effects)
for y, name in enumerate(names[::-1]):
    r = effects[name]["r"]
    n = effects[name]["n"]
    z = float(np.arctanh(r))
    se = 1 / math.sqrt(max(n - 3, 1))
    lo, hi = float(np.tanh(z - 1.96 * se)), float(np.tanh(z + 1.96 * se))
    ax.plot([lo, hi], [y, y], color=PALETTE["lav"], lw=3)
    ax.scatter([r], [y], color=PALETTE["violet"], s=52, zorder=3)
    ax.text(hi + 0.01, y, f"r = {r:+.3f} · n = {n:,}".replace(",", " "),
            va="center", fontsize=8)
ax.axvline(0, color=PALETTE["grey"], ls=":", lw=1)
ax.set_yticks(range(len(names)), names[::-1])
ax.set_xlabel("r: связь активности и ИМТ")
ax.set_title("Гармонизация: направление эффекта совпадает")
ax.grid(axis="y", visible=False)
fig.tight_layout()
save_fig(fig, "harmonized_effect")
plt.show()

<a id="sec-7-1"></a>
## 7.1. Главные компоненты и корреляции по наборам

Для каждого набора — своя корреляционная матрица и своя PCA. Сравнивать компоненты
между наборами нельзя: разные переменные и шкалы. Зато видно, какие показатели ходят
вместе внутри источника, и сколько дисперсии держат первые компоненты.

Правила чтения:

- корреляции — Пирсон по полным строкам, поэтому n в подписи меньше размера набора;
- нагрузки PC1 показывают главное направление изменчивости (обычно возраст и ИМТ);
- если PC1+PC2 объясняют меньше половины дисперсии, одним индексом набор не описать;
- AI4FoodDB — один участник: корреляции и PCA не определены, набор идёт как схема.

In [ ]:
def pca_bundle(df: pd.DataFrame, cols: list[str], name: str, color: str):
    """Корреляции, scree и нагрузки PC1–PC2 для одного набора."""
    d = df[cols].apply(pd.to_numeric, errors="coerce").dropna()
    if len(d) < 30:
        print(f"{name}: n={len(d)} — мало строк, PCA не считаем")
        return None
    Z = StandardScaler().fit_transform(d)
    pca = PCA().fit(Z)
    corr = d.corr()

    fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2))
    ax = axes[0]
    im = ax.imshow(corr, cmap="PuOr", vmin=-1, vmax=1)
    ax.set_xticks(range(len(cols)), cols, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(cols)), cols, fontsize=8)
    ax.set_title(f"{name}: корреляции")
    fig.colorbar(im, ax=ax, fraction=.046)

    ax = axes[1]
    ratios = pca.explained_variance_ratio_
    ax.bar(range(1, len(ratios) + 1), ratios, color=PALETTE[color])
    ax.plot(range(1, len(ratios) + 1), np.cumsum(ratios),
            color=PALETTE["dark"], marker="o", ms=3)
    ax.set_xlabel("компонента")
    ax.set_ylabel("доля дисперсии")
    ax.set_title(f"{name}: вклад компонент")

    ax = axes[2]
    load = pd.DataFrame(pca.components_[:2].T, index=cols, columns=["PC1", "PC2"])
    im = ax.imshow(load.to_numpy(), cmap="PuOr", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks([0, 1], ["PC1", "PC2"])
    ax.set_yticks(range(len(cols)), cols, fontsize=8)
    for i in range(len(cols)):
        for j in range(2):
            ax.text(j, i, f"{load.iloc[i, j]:+.2f}", ha="center", va="center", fontsize=7)
    ax.set_title(f"{name}: нагрузки PC1–PC2")
    fig.colorbar(im, ax=ax, fraction=.046)
    fig.tight_layout()
    save_fig(fig, f"pca_{color}")
    plt.show()

    pairs = corr.where(~np.eye(len(cols), dtype=bool)).stack()
    top = pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(3)
    print(f"{name}: n={len(d)}, PC1 = {ratios[0]:.0%}, PC1+PC2 = {ratios[:2].sum():.0%}")
    print("сильные связи:", ", ".join(f"{a}–{b} r={v:+.2f}" for (a, b), v in top.items()))
    return {"n": len(d), "pc1": float(ratios[0]), "pc2": float(ratios[1])}

brfss_pca = brfss.copy()
brfss_pca["ment"] = brfss_pca["MENTHLTH"].where(brfss_pca["MENTHLTH"].between(0, 30))
pca_brfss = pca_bundle(brfss_pca, ["bmi", "active", "ment", "_AGEG5YR", "_INCOMG1"],
                       "BRFSS", "violet")
pca_nhanes = pca_bundle(cohort, ["bmi", "sleep_h", "pa_min", "phq9",
                                 "DR1TKCAL", "DR1TFIBE", "DR1TSODI", "RIDAGEYR"],
                        "NHANES", "olive")
print("AI4FoodDB: один участник — корреляции и PCA не определены, набор идёт как схема.")

<a id="sec-8"></a>
## 8. QC: пропуски, диапазоны, выбросы

Минимум перед моделью:

1. таблица пропусков по блокам (доля непустых);
2. проверка диапазонов (ИМТ 10–80, сон 0–24, возраст 18–80);
3. выбросы — не удалять молча: либо правило (например, |z| > 4), либо решение в журнале;
4. дубликаты по ключу (`SEQN` в NHANES) — проверять всегда.

Правило: каждая цифра в отчёте должна иметь ячейку, которая её считает.

In [ ]:
def missing_report(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rep = pd.DataFrame({
        "непустых": df[cols].notna().sum(),
        "доля": df[cols].notna().mean().round(3),
        "мин": df[cols].min(numeric_only=True),
        "макс": df[cols].max(numeric_only=True),
    })
    return rep.sort_values("доля")

def check_ranges(df: pd.DataFrame, rules: dict[str, tuple[float, float]]) -> pd.DataFrame:
    out = []
    for col, (lo, hi) in rules.items():
        s = df[col].dropna()
        bad = int(((s < lo) | (s > hi)).sum())
        out.append({"колонка": col, "правило": f"{lo}–{hi}", "вне": bad,
                    "мин": float(s.min()) if len(s) else None,
                    "макс": float(s.max()) if len(s) else None})
    return pd.DataFrame(out)

missing_report(brfss, ["bmi", "active", "GENHLTH", "MENTHLTH"])

check_ranges(brfss, {"bmi": (10, 80), "MENTHLTH": (0, 30)})

In [ ]:
print("NHANES: дубликаты SEQN:", int(cohort["SEQN"].duplicated().sum()))
print("BRFSS: дубликатов нет по построению (одна строка = респондент)")

nh_cols = ["bmi", "sleep_h", "phq9", "DR1TKCAL", "BPXOSY1", "BPXOPLS1"]
missing_report(cohort, nh_cols)

check_ranges(cohort, {"bmi": (10, 80), "sleep_h": (0, 24), "phq9": (0, 27),
                      "BPXOSY1": (60, 260), "DR1TKCAL": (0, 8000)})

In [ ]:
# QC: заполненность переменных, которые идут в модель.
qc = pd.concat([
    pd.DataFrame({"источник": "BRFSS", "переменная": ["bmi", "active", "MENTHLTH"],
                  "доля": [brfss[c].notna().mean() for c in ["bmi", "active", "MENTHLTH"]]}),
    pd.DataFrame({"источник": "NHANES", "переменная": ["bmi", "sleep_h", "pa_min", "phq9", "DR1TKCAL"],
                  "доля": [cohort[c].notna().mean() for c in ["bmi", "sleep_h", "pa_min", "phq9", "DR1TKCAL"]]}),
])
fig, ax = plt.subplots(figsize=(7.6, 3.8))
colors = {"BRFSS": PALETTE["violet"], "NHANES": PALETTE["olive"]}
for i, row in qc.reset_index(drop=True).iterrows():
    ax.barh(i, row["доля"], color=colors[row["источник"]], height=.7)
    ax.text(min(row["доля"] + .02, .97), i, f"{row['доля']:.0%}", va="center", fontsize=8)
ax.set_yticks(range(len(qc)), qc["источник"] + " · " + qc["переменная"], fontsize=8)
ax.set_xlim(0, 1)
ax.set_xlabel("доля непустых")
ax.set_title("QC: заполненность переменных до модели")
ax.grid(axis="y", visible=False)
fig.tight_layout()
save_fig(fig, "qc_missing")
plt.show()

<a id="sec-9"></a>
## 9. Модель: важность блоков привычек

Гипотеза: привычки (сон, активность, рацион, дистресс) объясняют ИМТ сверх демографии.
Считаем не важность отдельных признаков, а **блоков**: переставляем все колонки блока
и смотрим, насколько падает R². Это устойчивее и честнее, чем «топ-10 признаков».

Правила:

- фиксируем три seed'а и показываем разброс, а не один результат;
- bootstrap по тестовым строкам даёт 95 % CI важности блока;
- если CI блоков пересекаются, ранг блока — не вывод, а гипотеза;
- тестовая выборка не участвует в обучении.

In [ ]:
BLOCKS = {
    "demo": ["RIDAGEYR", "RIAGENDR", "DMDEDUC2", "INDFMPIR"],
    "diet": ["DR1TKCAL", "DR1TPROT", "DR1TCARB", "DR1TTFAT", "DR1TFIBE", "DR1TSODI"],
    "sleep": ["sleep_h"],
    "activity": ["pa_min"],
    "distress": ["phq9"],
}
TARGET = "bmi"

model_df = cohort[["SEQN", TARGET] + [c for cols in BLOCKS.values() for c in cols]]
model_df = model_df.dropna(subset=[TARGET]).copy()
model_df[[c for cols in BLOCKS.values() for c in cols]] = \
    model_df[[c for cols in BLOCKS.values() for c in cols]].fillna(
        model_df[[c for cols in BLOCKS.values() for c in cols]].median()).fillna(0)

X = model_df[[c for cols in BLOCKS.values() for c in cols]]
y = model_df[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEEDS[0])

def fit_forest(seed: int) -> RandomForestRegressor:
    m = RandomForestRegressor(n_estimators=250, min_samples_leaf=5,
                              n_jobs=-1, random_state=seed)
    return m.fit(X_tr, y_tr)

def block_drops(model, Xd: pd.DataFrame, yd: pd.Series,
                blocks: dict[str, list[str]], n_perm: int = 5, seed: int = 42) -> dict[str, float]:
    """Падение R² при перестановке колонок блока."""
    rng = np.random.default_rng(seed)
    base = r2_score(yd, model.predict(Xd))
    out = {}
    for name, cols in blocks.items():
        drops = []
        for _ in range(n_perm):
            Xp = Xd.copy()
            for c in cols:
                Xp[c] = rng.permutation(Xp[c].to_numpy())
            drops.append(base - r2_score(yd, model.predict(Xp)))
        out[name] = float(np.mean(drops))
    return out

forest = fit_forest(SEEDS[0])
print("R² на тесте:", round(r2_score(y_te, forest.predict(X_te)), 3))
base_drops = block_drops(forest, X_te, y_te, BLOCKS)
pd.Series(base_drops).sort_values(ascending=False).round(4)

In [ ]:
def bootstrap_block_ci(model, Xd: pd.DataFrame, yd: pd.Series,
                       blocks: dict[str, list[str]], n_boot: int = 60,
                       seed: int = 42) -> pd.DataFrame:
    """95 % CI важности блоков: bootstrap по строкам теста."""
    rng = np.random.default_rng(seed)
    idx = np.arange(len(Xd))
    samples = {name: [] for name in blocks}
    for _ in range(n_boot):
        take = rng.choice(idx, size=len(idx), replace=True)
        drops = block_drops(model, Xd.iloc[take], yd.iloc[take], blocks,
                            n_perm=3, seed=int(rng.integers(0, 10**6)))
        for name, v in drops.items():
            samples[name].append(v)
    rows = []
    for name, vals in samples.items():
        arr = np.asarray(vals)
        rows.append({"блок": name, "важность": float(np.mean(arr)),
                     "lo": float(np.percentile(arr, 2.5)),
                     "hi": float(np.percentile(arr, 97.5))})
    return pd.DataFrame(rows).sort_values("важность", ascending=False)

ci = bootstrap_block_ci(forest, X_te, y_te, BLOCKS)
ci.round(4)


# Расшифровка блоков: из каких переменных собран каждый фактор.
BLOCK_LABELS = {
    "demo": "демография",
    "diet": "питание",
    "sleep": "сон",
    "activity": "активность",
    "distress": "дистресс",
}
FEATURE_DOCS = {
    "RIDAGEYR": ("возраст участника", "годы"),
    "RIAGENDR": ("пол", "1 — муж, 2 — жен"),
    "DMDEDUC2": ("образование взрослого", "1–5, код"),
    "INDFMPIR": ("отношение дохода семьи к порогу бедности", "индекс"),
    "DR1TKCAL": ("энергия за 24 ч", "ккал"),
    "DR1TPROT": ("белок за 24 ч", "г"),
    "DR1TCARB": ("углеводы за 24 ч", "г"),
    "DR1TTFAT": ("жиры за 24 ч", "г"),
    "DR1TFIBE": ("клетчатка за 24 ч", "г"),
    "DR1TSODI": ("натрий за 24 ч", "мг"),
    "sleep_h": ("часы сна (SLD012)", "ч/ночь"),
    "pa_min": ("активность за неделю (PAQ: PAD790/PAD810)", "мин/нед"),
    "phq9": ("PHQ-9, сумма девяти пунктов (DPQ010–090)", "баллы 0–27"),
}
rows = []
for block, cols in BLOCKS.items():
    for c in cols:
        what, unit = FEATURE_DOCS.get(c, (c, ""))
        rows.append({"блок": BLOCK_LABELS.get(block, block), "переменная": c,
                     "что это": what, "единицы": unit})
decoding = pd.DataFrame(rows)
print("блоков:", len(BLOCKS), "| переменных:", len(decoding))
decoding

In [ ]:
# H7: устойчивость ранга на трёх seed'ах.
rank_rows = []
for seed in SEEDS:
    m = fit_forest(seed)
    drops = block_drops(m, X_te, y_te, BLOCKS, seed=seed)
    order = sorted(drops, key=drops.get, reverse=True)
    rank_rows.append({"seed": seed, **{k: i + 1 for i, k in enumerate(order)}})
ranks = pd.DataFrame(rank_rows).set_index("seed")
ranks

In [ ]:
# Модель: разброс R² по seed'ам и устойчивость рангов блоков.
r2 = {s: float(r2_score(y_te, fit_forest(s).predict(X_te))) for s in SEEDS}
fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.5))
ax = axes[0]
ax.bar([str(s) for s in SEEDS], [r2[s] for s in SEEDS], color=PALETTE["violet"])
ax.set_xlabel("random_state")
ax.set_ylabel("R² на тесте")
ax.set_title("R² по seed'ам")
for i, s in enumerate(SEEDS):
    ax.text(i, r2[s] + .002, f"{r2[s]:.3f}", ha="center", fontsize=8)
ax = axes[1]
im = ax.imshow(ranks.to_numpy(), cmap="Purples_r", aspect="auto")
ax.set_xticks(range(len(ranks.columns)), ranks.columns, rotation=25, ha="right", fontsize=8)
ax.set_yticks(range(len(ranks.index)), [f"seed {s}" for s in ranks.index], fontsize=8)
for i in range(ranks.shape[0]):
    for j in range(ranks.shape[1]):
        ax.text(j, i, int(ranks.iloc[i, j]), ha="center", va="center", fontsize=9)
ax.set_title("Ранг блоков по seed'ам")
fig.colorbar(im, ax=ax, fraction=.04)
fig.tight_layout()
save_fig(fig, "model_seeds")
plt.show()

<a id="sec-10"></a>
## 10. Метаанализ: эффект по наборам

Метаанализ нужен, когда один и тот же эффект измерен в разных наборах. У нас два
независимых источника (BRFSS и NHANES) и один набор-схема (AI4FoodDB, n=1) — он в
объединение не идёт.

Правила:

- переводим r в z Фишера: `z = atanh(r)`, `SE = 1 / sqrt(n - 3)`;
- fixed effect взвешивает по `1/SE²`; random effect (DerSimonian–Laird) добавляет τ²;
- I² — доля разброса сверх случайного; при двух исследованиях трактуем осторожно;
- forest plot обязателен: точка, CI, вес, итоговая строка.

Если эффекты разнонаправлены, объединение не спасает — это вывод, а не ошибка.

In [ ]:
def fisher(r: float, n: int) -> tuple[float, float]:
    """r -> z Фишера и его стандартная ошибка."""
    return float(np.arctanh(r)), float(1 / math.sqrt(max(n - 3, 1)))

def meta_fixed(zs: np.ndarray, ses: np.ndarray) -> dict:
    w = 1 / ses**2
    est = float((w * zs).sum() / w.sum())
    se = float(math.sqrt(1 / w.sum()))
    return {"est": est, "se": se}

def meta_random(zs: np.ndarray, ses: np.ndarray) -> dict:
    fixed = meta_fixed(zs, ses)
    q = float((1 / ses**2 * (zs - fixed["est"])**2).sum())
    dfree = len(zs) - 1
    c = float((1 / ses**2).sum() - ((1 / ses**2)**2).sum() / (1 / ses**2).sum())
    tau2 = max(0.0, (q - dfree) / c) if c > 0 else 0.0
    w = 1 / (ses**2 + tau2)
    est = float((w * zs).sum() / w.sum())
    se = float(math.sqrt(1 / w.sum()))
    i2 = max(0.0, (q - dfree) / q) * 100 if q > 0 else 0.0
    return {"est": est, "se": se, "tau2": tau2, "q": q, "i2": i2}

names = list(effects)
zs = np.array([fisher(effects[n]["r"], effects[n]["n"])[0] for n in names])
ses = np.array([fisher(effects[n]["r"], effects[n]["n"])[1] for n in names])
fixed = meta_fixed(zs, ses)
random = meta_random(zs, ses)
summary = pd.DataFrame({
    "набор": names,
    "n": [effects[n]["n"] for n in names],
    "r": [effects[n]["r"] for n in names],
    "z": zs, "se": ses,
})
print(summary.round(4).to_string(index=False))
print(f"fixed: r={np.tanh(fixed['est']):.3f} "
      f"({np.tanh(fixed['est'] - 1.96 * fixed['se']):.3f}…{np.tanh(fixed['est'] + 1.96 * fixed['se']):.3f})")
print(f"random: r={np.tanh(random['est']):.3f}, I²={random['i2']:.0f} %, τ²={random['tau2']:.4f}")

In [ ]:
# Forest plot: точка — r, усы — 95 % CI, ромб — итог random effects.
fig, ax = plt.subplots(figsize=(7.2, 2.8))
ys = np.arange(len(names))[::-1]
for y, name in zip(ys, names):
    r = effects[name]["r"]
    z, se = fisher(r, effects[name]["n"])
    lo, hi = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
    ax.plot([lo, hi], [y, y], color=PALETTE["lav"], lw=2, zorder=1)
    ax.scatter([r], [y], s=46, color=PALETTE["violet"], zorder=3)
    ax.text(hi + 0.01, y, f"{r:+.2f} (n={effects[name]['n']:,})".replace(",", " "),
            va="center", fontsize=9)

r_hat = float(np.tanh(random["est"]))
lo_r = float(np.tanh(random["est"] - 1.96 * random["se"]))
hi_r = float(np.tanh(random["est"] + 1.96 * random["se"]))
ax.axvline(0, color=PALETTE["grey"], lw=1, ls=":")
ax.errorbar([r_hat], [-1], xerr=[[r_hat - lo_r], [hi_r - r_hat]], fmt="D",
            color=PALETTE["magenta"], capsize=4, ms=7)
ax.set_yticks(list(ys) + [-1], names + ["random effects"])
ax.set_xlabel("r: связь активности и ИМТ")
ax.set_title("Метаанализ: активность и ИМТ, два независимых набора")
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(FIG / "forest_activity_bmi.png", bbox_inches="tight")
plt.show()

<a id="sec-11"></a>
## 11. Кластерный анализ: когорты по привычкам

Кластеризация не доказывает причинность, но показывает, что популяция неоднородна:
разные профили привычек дают разные риски. Кластеризуем когорту NHANES по восьми
признакам (возраст, ИМТ, сон, активность, PHQ-9, ккал, клетчатка, натрий).

Правила:

- признаки стандартизуем, иначе возраст и ккал перевесят всё остальное;
- силуэт считаем для k = 2…6: он показывает, где разделение формально чётче;
- для сегментации фиксируем **k = 4**: две когорты слишком грубы для продукта,
  а при четырёх видно разные точки входа (ИМТ, возраст, активность);
- профиль каждого кластера обязателен — без него кластеры не интерпретируются;
- область когорты на графике — эллипс 1.8σ по ковариации: он устойчивее выпуклой
  оболочки, которую вытягивают единичные выбросы;
- имена когорт собираются из отклонений профиля от среднего и проверяются человеком.


In [ ]:
FEATURES = ["RIDAGEYR", "bmi", "sleep_h", "pa_min", "phq9",
            "DR1TKCAL", "DR1TFIBE", "DR1TSODI"]
cl = cohort[FEATURES].dropna().copy()
X = StandardScaler().fit_transform(cl)
print("строк для кластеризации:", len(cl), "| признаков:", len(FEATURES))

rows = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=20, random_state=42).fit(X)
    rows.append({"k": k, "silhouette": silhouette_score(X, km.labels_),
                 "inertia": km.inertia_})
sil = pd.DataFrame(rows)
sil.round(3)

In [ ]:
K = 4  # ручной выбор: силуэт предпочитает меньше кластеров, для сегментации нужно 4
km = KMeans(n_clusters=K, n_init=50, random_state=42).fit(X)
cl["cluster"] = km.labels_

profile = cl.groupby("cluster").agg(
    n=("bmi", "size"),
    возраст=("RIDAGEYR", "mean"),
    имт=("bmi", "mean"),
    сон_ч=("sleep_h", "mean"),
    активность_мин=("pa_min", "median"),
    phq9=("phq9", "mean"),
    ккал=("DR1TKCAL", "mean"),
    клетчатка=("DR1TFIBE", "mean"),
).round(1)

print(sil.round(3).to_string(index=False))
print("выбрано кластеров:", K,
      "· лучший силуэт при k =", int(sil.sort_values("silhouette").iloc[-1]["k"]))
profile


In [ ]:
# Кластеры: выбор k по silhouette и профиль когорт в z-оценках.
fig, axes = plt.subplots(1, 2, figsize=(11.2, 3.9))
ax = axes[0]
ax.plot(sil["k"], sil["silhouette"], marker="o", color=PALETTE["violet"])
ax.axvline(K, color=PALETTE["magenta"], ls="--", lw=1)
ax.set_xlabel("число кластеров k")
ax.set_ylabel("silhouette")
ax.set_title("Выбор k по silhouette")
ax = axes[1]
prof = profile.drop(columns=["n"]).copy()
prof = (prof - prof.mean()) / prof.std(ddof=0).replace(0, 1)
im = ax.imshow(prof.to_numpy(), cmap="PuOr", aspect="auto", vmin=-2, vmax=2)
ax.set_xticks(range(len(prof.columns)), prof.columns, rotation=35, ha="right", fontsize=8)
ax.set_yticks(range(len(prof.index)), [f"когорта {c}" for c in prof.index], fontsize=8)
for i in range(prof.shape[0]):
    for j in range(prof.shape[1]):
        ax.text(j, i, f"{prof.iloc[i, j]:+.1f}", ha="center", va="center", fontsize=7)
ax.set_title("Профиль когорт (z-оценки)")
fig.colorbar(im, ax=ax, fraction=.04)
fig.tight_layout()
save_fig(fig, "cluster_profile")
plt.show()

In [ ]:
from matplotlib.patches import Ellipse

base = cl[["RIDAGEYR", "bmi", "sleep_h", "pa_min", "phq9"]].mean()


def cohort_name(row) -> str:
    """Короткое имя когорты по отклонению профиля от среднего."""
    parts = []
    if row["имт"] >= base["bmi"] + 1.5:
        parts.append("высокий ИМТ")
    elif row["имт"] <= base["bmi"] - 1.5:
        parts.append("низкий ИМТ")
    if row["возраст"] >= base["RIDAGEYR"] + 7:
        parts.append("старше")
    elif row["возраст"] <= base["RIDAGEYR"] - 7:
        parts.append("моложе")
    if row["активность_мин"] >= base["pa_min"] * 1.5:
        parts.append("активные")
    elif row["активность_мин"] <= base["pa_min"] * 0.5:
        parts.append("мало активности")
    if row["phq9"] >= base["phq9"] + 2:
        parts.append("выше дистресс")
    if row["сон_ч"] <= base["sleep_h"] - 0.7:
        parts.append("короткий сон")
    return ", ".join(parts) or "средний профиль"


names = {int(c): cohort_name(profile.loc[c]) for c in profile.index}
cl["cohort"] = cl["cluster"].map(names)
print("когорты:", {f"{i + 1}": names[c] for i, c in enumerate(sorted(names))})

proj = PCA(n_components=2, random_state=42).fit_transform(X)
cl["pc1"], cl["pc2"] = proj[:, 0], proj[:, 1]
colors = [PALETTE["violet"], PALETTE["magenta"], PALETTE["olive"], PALETTE["coral"],
          PALETTE["lav"]]

fig, ax = plt.subplots(figsize=(9.4, 5.8))
for i, c in enumerate(sorted(cl["cluster"].unique())):
    sub = cl[cl["cluster"] == c]
    color = colors[i % len(colors)]
    label = f"{i + 1}. {names[int(c)]} · n={len(sub):,}".replace(",", " ")
    ax.scatter(sub["pc1"], sub["pc2"], s=7, alpha=.28, color=color, label=label)

    # Область когорты — эллипс 1.8σ по ковариации: устойчивее выпуклой оболочки,
    # которую вытягивают единичные выбросы.
    cov = np.cov(sub[["pc1", "pc2"]].to_numpy().T)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = float(np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0])))
    width, height = 2 * 1.8 * np.sqrt(np.maximum(vals, 0))
    ax.add_patch(Ellipse((sub["pc1"].mean(), sub["pc2"].mean()), width, height,
                         angle=angle, facecolor=color, edgecolor=color,
                         alpha=.12, lw=1.6))

    cx, cy = sub["pc1"].mean(), sub["pc2"].mean()
    ax.scatter(cx, cy, marker="D", s=84, color=color, edgecolor="white", zorder=5)
    ax.text(cx, cy, str(i + 1), ha="center", va="center", fontsize=8,
            fontweight="bold", color="white", zorder=6)

ax.set_xlabel("главная компонента 1")
ax.set_ylabel("главная компонента 2")
ax.set_title(f"Когорты по привычкам: k={K}, область — эллипс 1.8σ")
ax.legend(markerscale=1.8, fontsize=8, loc="best")
fig.tight_layout()
fig.savefig(FIG / "clusters_pca.png", bbox_inches="tight")
plt.show()
print("сохранено:", FIG / "clusters_pca.png")


**Как читать.** Точка — человек, цвет и эллипс — когорта, цифра в ромбе — номер
когорты. Оси — главные компоненты: их нельзя назвать одной переменной, но близость точек
означает похожий профиль.

**Что важно при k = 4.** Силуэт формально выше при k = 2 (0.20 против 0.17): когорты
пересекаются, границы условные. Это не ошибка, а цена более полезной сегментации:
при четырёх группах видно, что «высокий ИМТ» и «старше» — разные люди, а активные
(их меньше всего) отличаются от среднего профиля. Для продукта важнее профиль и точка
входа, а не чёткая граница на графике.

**Что делать с этим.** Когорты — не диагноз, а сегменты: у одной группы точка входа —
активность, у другой — сон или питание. Имена в легенде собраны по отклонениям от
среднего; команда проверяет их на профиле и защищает на ревью.


<a id="sec-12"></a>
## 12. Стандарт визуализации

Один график — одно утверждение. Оси подписаны с единицами, легенда не дублирует
подписи, цвет несёт смысл, а не украшение.

Палитра практики: фиолетовый — основной ряд, маджента — акцент или итог,
олива — «в норме», коралл — предупреждение. Серый — фон и ориентиры.

Правила:

- заголовок отвечает на вопрос, а не называет переменную;
- подпись оси — с единицей измерения (`ИМТ, кг/м²`, `часы сна`);
- не больше двух цветов на один смысл;
- сохраняем в `out/figures/` с dpi 160 и именем `что_где.png`;
- интерактив — только если он что-то добавляет; для отчёта хватает PNG.

In [ ]:
def save_fig(fig, name: str) -> Path:
    """Сохранить график в out/figures и вернуть путь."""
    path = FIG / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    return path

def bar_blocks(ci_df: pd.DataFrame, name: str = "block_importance_bmi"):
    fig, ax = plt.subplots(figsize=(6.4, 3.0))
    order = ci_df.sort_values("важность")
    ypos = np.arange(len(order))
    ax.barh(ypos, order["важность"], color=PALETTE["violet"], height=0.6)
    ax.errorbar(order["важность"], ypos,
                xerr=[order["важность"] - order["lo"], order["hi"] - order["важность"]],
                fmt="none", ecolor=PALETTE["dark"], capsize=3, lw=1.1)
    ax.set_yticks(ypos, [BLOCK_LABELS.get(b, b) for b in order["блок"]])
    ax.set_xlabel("падение R² при перестановке блока")
    ax.set_title("Важность блоков привычек для ИМТ")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    return save_fig(fig, name)

print("сохранено:", bar_blocks(ci))

<a id="sec-13"></a>
## 13. Документирование и воспроизводимость

Код без документации — это одноразовый эксперимент. Минимальный стандарт:

- у каждой функции docstring: что делает, что на входе, что на выходе;
- `from __future__ import annotations` и типы аргументов;
- все случайности через `random_state` / `np.random.default_rng(seed)`;
- версии библиотек и дата прогона пишутся в `out/notebook_run.json`;
- ячейки идут сверху вниз без ручных правок: «Run All» должен давать тот же результат;
- выводы в markdown, а не в комментариях к коду.

In [ ]:
import sklearn

run = {
    "date": datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "versions": {"pandas": pd.__version__, "numpy": np.__version__,
                 "sklearn": sklearn.__version__,
                 "matplotlib": matplotlib.__version__,
                 "pyarrow": __import__("pyarrow").__version__},
    "seeds": SEEDS,
    "rows": {"brfss": int(len(brfss)), "nhanes_cohort": int(len(cohort)),
             "ai4fooddb_tables": len(a4f_files)},
    "effects": effects,
    "meta": {"random_r": r_hat, "i2": random["i2"]},
    "figures": sorted(p.name for p in FIG.glob("*.png")),
}
(OUT / "notebook_run.json").write_text(json.dumps(run, ensure_ascii=False, indent=1),
                                       encoding="utf-8")
print(json.dumps(run, ensure_ascii=False, indent=1)[:700])

<a id="sec-14"></a>
## 14. Чек-лист перед сдачей

| Проверка | Критерий |
|---|---|
| Данные | размеры совпадают с `PASSPORT.md`, sha256 не менялись |
| Коды | отказы и «не знаю» переведены в пропуски, а не в числа |
| Веса | BRFSS и NHANES считаются с весом; невзвешенные — только как сравнение |
| Пропуски | таблица по блокам, решение по каждому блоку записано |
| Модель | train/test разделены, три seed'а, CI у важности блоков |
| Метаанализ | r → z, fixed и random, I², forest plot |
| Графики | подписи с единицами, палитра, сохранение в `out/figures/` |
| Код | docstring, типы, seed, «Run All» без ручных шагов |
| Отчёт | каждая цифра имеет ячейку-источник; выводы отдельно от кода |
| Лицензии | AI4FoodDB не распространяется, parquet не публикуется |

**Что дальше.** Команда копирует ноутбук в `notebooks/<team>_<task>.ipynb`, меняет
блоки под свою гипотезу и приносит на ревью: сначала QC и описательные таблицы,
потом модель, потом метаанализ.

<a id="sec-15"></a>
## 15. Выводы: как исследование решает проблему

**Что показали данные.**

- Привычки объясняют ИМТ слабее, чем ожидается: R² модели на тесте около 0.04, то есть
  сон, активность, рацион и дистресс вместе дают несколько процентов разброса. Это
  честный результат, а не повод подгонять модель.
- Эффект активности устойчив и совпадает по направлению в двух независимых наборах:
  BRFSS r ≈ −0.14 и NHANES r ≈ −0.15. Метаанализ даёт r ≈ −0.14 и I² = 0.
- Возраст и ИМТ тянут на себя большую часть сигнала: без них вклад привычек почти
  не виден. Поэтому важность блоков считается сверх демографии.

**Что даёт кластерный анализ.**

- Популяция неоднородна: кластеры различаются возрастом, ИМТ и активностью, а не
  только рационом. Одна модель «на всех» сглаживает эти различия.
- Области на диаграмме показывают, где заканчивается один профиль и начинается другой;
  это основа для сегментации в продукте.

**Как это решает проблему.**

1. Данные трёх типов сведены в один конвейер: вопрос «что с привычками» больше не
   требует трёх отдельных исследований.
2. Появился измеримый ответ: активность — самый сильный из привычных блоков, но эффект
   мал; ждать от приложения про сон или питание изменения ИМТ в одиночку нельзя.
3. Когорты дают продуктовую рамку: разные сегменты, разные точки входа и разные метрики.
4. Воспроизводимость зафиксирована: спецификация, QC, seed'ы, метрики и графики
   пересобираются одной командой.

**Ограничения.**

- BRFSS — самоотчёт, NHANES — один цикл и 24h recall, AI4FoodDB — один участник.
- Наблюдательные данные не дают причинности: направление связи может быть обратным.
- Часть переменных (сон, рацион) измерена разными инструментами, сравнение идёт по
  направлению эффекта, а не по абсолютным значениям.

**Следующий шаг.** Расширить наборы (BRFSS за несколько лет, NHANES 2017–2020), добавить
метарегрессию по возрасту и полу, проверить устойчивость кластеров на подвыборках.